In [1]:
import os
from langgraph.graph import StateGraph, END
from ai_framework.nodes import *
# from ai_framework.nodes import completedProcess,thinking_steps,generate_answer

c:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def route_after_validation(state: GraphState):
    if state["is_valid"]:
        return "thinking"
    else:
        return "end"

def build_graph():
    builder = StateGraph(GraphState)
    builder.add_node("validate", validate_question)
    builder.add_node("Thinking", thinking_steps)
    builder.add_node("retrive_document", generate_answer)
    builder.add_node("Websearch", websearch)
    builder.add_node("suggest_questions", suggest_questions)
    builder.set_entry_point("validate")
    builder.add_conditional_edges(
        "validate",
        route_after_validation,
        {
            "thinking": "Thinking",
            "end":END
        }
    )
    builder.add_edge("Thinking", "retrive_document")
    builder.add_edge("retrive_document", "Websearch")
    builder.add_edge("Websearch", "suggest_questions")
    builder.add_edge("suggest_questions", END)
    return builder.compile()

# builder.add_node("websearch", websearch)
    # builder.add_node("followupquestion", suggest_questions)

In [3]:
# -----------------------------
# STREAM FUNCTION
# -----------------------------
def ask_question_stream(query: str):
    graph = build_graph()
    print("\n=== STREAMING START ===\n")
    # final_state = {}
    inputObj={"query": query,
              "filename":"reactaa.pdf",
              "messageId":"1234",
              "user_id":"21a1ff59-f04b-450e-bf46-322617dae796",  #session['user_id'] 
              "user_name":"pranay", #session['user_name'] 
              "filename":'reactaa.pdf',
              "description":"this pdf is about react javascript framework"
              }

    for step in graph.stream(inputObj):
        # print('step',step)
        for node, output in step.items():
            print("-"*10)
            print("node", node)
            print( output)
            print("-"*10)
            # final_state.update(output)
    print("\n=== STREAMING END ===\n")
    # return final_state


In [4]:
result=ask_question_stream("What is the significance of keys in React?")
result


=== STREAMING START ===

----------
node validate
{'is_valid': True}
----------
----------
node Thinking
{'thinking': {'content': '{\n    "Steps": [\n        "Step 1": "First, I will retrieve relevant documents related to React and its components.",\n        "Step 2": "Next, I will use RAG to search for information about \'keys\' within the retrieved documents, focusing on their role and significance in React.",\n        "Step 3": "Then, I will analyze the extracted information to understand the purpose and benefits of using keys in React.",\n        "Step 4": "Finally, I', 'used_tokens': 100, 'prompt_tokens': 158, 'total_tokens': 258, 'messageid': '1234'}}
----------
----------
node retrive_document
{'documentAnswer': {'content': '{\n    "answer": "Keys are used to uniquely identify and differentiate between components in React, helping React identify which items have changed, added, or removed, and efficiently update the DOM when the list changes.",\n    "citations": ["reactaa.pdf, 